# Explore image-parameter results

This notebook reads the summary CSV produced by `plot_image_params_results.py`; it does **not** open or process FITS files. Set `CSV_PATH` below to the existing `image_parameter_summary.csv`. Once FITS extraction has been done and the CSV is accessible, plotting is light enough to do locally without Slurm.

The notebook needs `pandas`, `matplotlib`, and `ipywidgets` (as well as `numpy`). If the controls do not render, enable/install the Jupyter widgets support in your notebook environment.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Update this to the summary CSV written by the FITS extraction run.
CSV_PATH = Path("parameter_plots/image_parameter_summary.csv")


In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Summary CSV not found: {CSV_PATH.resolve()}")

results = pd.read_csv(CSV_PATH)
required_columns = {
    "obsnum", "robust", "minuv", "tukey", "rms_mjy_beam",
    "n_components_all", "n_components_flags0",
}
missing = required_columns - set(results.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {', '.join(sorted(missing))}")

for column in ("robust", "minuv", "tukey", "rms_mjy_beam",
               "n_components_all", "n_components_flags0"):
    results[column] = pd.to_numeric(results[column], errors="raise")
results["obsnum"] = results["obsnum"].astype(str)

print(f"Loaded {len(results):,} runs from {CSV_PATH.resolve()}")
results.head()


## Interactive plots

Choose an observation in the first cell, then edit `X_LIMITS` in the next cell. Each entry sets the horizontal-axis limits for that parameter's plot column; use `None` for automatic limits. These are display limits only—the CSV is unchanged. Re-run the plotting cell after changing `X_LIMITS`.

In [ ]:
observations = sorted(results["obsnum"].unique())
observation = widgets.Dropdown(
    options=[("All observations", "All")] + [(obs, obs) for obs in observations],
    value="All", description="Observation:", layout=widgets.Layout(width="350px"),
)
count_metric = widgets.Dropdown(
    options=[("Components with flags = 0", "n_components_flags0"),
             ("All catalogue components", "n_components_all")],
    value="n_components_flags0", description="Counts:",
    layout=widgets.Layout(width="350px"),
)
display(widgets.VBox([observation, count_metric]))


In [ ]:
# EDIT THESE LIMITS: (left, right) for each plot column; None means automatic.
# Example: show robust from -0.5 to 0.3 and minuv up to 100.
X_LIMITS = {
    "robust": (-0.5, 0.3),
    "minuv": (None, 100),
    "tukey": None,
}


In [ ]:
PARAMS = ("robust", "minuv", "tukey")
LABELS = {
    "robust": "Briggs robust",
    "minuv": "Minimum UV distance (wavelengths)",
    "tukey": "Inner Tukey taper (wavelengths)",
}

def make_plot(obs_choice, metric):
    selected = results if obs_choice == "All" else results[results.obsnum == obs_choice]
    if selected.empty:
        raise ValueError("No rows match the selected observation.")

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    for col, param in enumerate(PARAMS):
        other_params = [name for name in PARAMS if name != param]
        group_columns = other_params + ([] if obs_choice != "All" else ["obsnum"])
        for axis, y_metric in zip(axes[:, col], ("rms_mjy_beam", metric)):
            for _, group in selected.groupby(group_columns, dropna=False):
                group = group.sort_values(param)
                axis.plot(group[param], group[y_metric], color="0.65",
                          linewidth=0.6, alpha=0.5)
            axis.scatter(selected[param], selected[y_metric], s=13,
                         alpha=0.5, color="tab:blue")
            medians = selected.groupby(param, as_index=False)[y_metric].median().sort_values(param)
            axis.plot(medians[param], medians[y_metric], "o-", color="tab:orange",
                      linewidth=2, label="Median over other settings")
            limits = X_LIMITS[param]
            if limits is not None:
                axis.set_xlim(left=limits[0], right=limits[1])
            axis.set_xlabel(LABELS[param])
            axis.grid(alpha=0.2)
        axes[0, col].set_ylabel("Median map RMS (mJy/beam)")
        axes[1, col].set_ylabel(
            "Components (flags = 0)" if metric == "n_components_flags0"
            else "All catalogue components"
        )
    axes[0, 0].legend(fontsize=8)
    title = "All observations" if obs_choice == "All" else f"Observation {obs_choice}"
    fig.suptitle(f"{title} — {len(selected)} evaluated combinations\n"
                 "Points: individual runs; grey lines: other settings fixed.", fontsize=12)
    return fig

plot_output = widgets.interactive_output(
    lambda obs_choice, metric: display(make_plot(obs_choice, metric)),
    {"obs_choice": observation, "metric": count_metric},
)
display(plot_output)


## Save a plot

To save the currently selected view, run the cell below. It writes a PNG into the current working directory. Change `PNG_PATH` if you want it somewhere else.

In [ ]:
PNG_PATH = Path("image_parameter_exploration.png")
fig = make_plot(observation.value, count_metric.value)
fig.savefig(PNG_PATH, dpi=160)
plt.close(fig)
print(f"Saved {PNG_PATH.resolve()}")
